# Corrective Retrieval-Augmented Generation (CRAG) Evaluation

This notebook benchmarks an end-to-end **Corrective RAG (CRAG)** pipeline.

### Core Pipeline Stages:
1. **Document Ingestion**: Parsing PDF, DOCX, PPTX, TXT into ChromaDB.
2. **Document Grader**: Evaluates retrieved chunks before generation.
3. **Query Rewriter**: Reformulates weak or ambiguous queries.
4. **Hallucination Prevention**: Refuses out-of-domain queries without hallucinating.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from crag_engine import CRAGConfig, CRAGEngine

config = CRAGConfig(
    data_dir=Path('sample_docs'),
    chroma_dir=Path('data/chroma_db'),
    relevance_threshold=0.35
)
engine = CRAGEngine(config)
print('CRAG Engine initialized successfully with device:', engine.device)


## 1. Knowledge Base Ingestion & Coverage
We ingest the academic documents and visualize chunk coverage.


In [ ]:
report = engine.ingest_files()
print(f"Ingestion Complete: {report['chunks']} chunks from {report['files']} files.")

all_meta = engine.collection.get(include=['metadatas'])['metadatas']
source_counts = pd.Series([m['source'] for m in all_meta]).value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
source_counts.plot(kind='bar', color='#1E3A8A', edgecolor='black', ax=ax)
ax.set_title('Indexed Knowledge Chunks per Document')
ax.set_ylabel('Number of Chunks')
plt.xticks(rotation=20, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## 2. Benchmark Evaluation Across Query Categories
We test the pipeline against direct, ambiguous, and out-of-domain questions.


In [ ]:
eval_df = pd.read_csv('evaluation_dataset.csv')
results = []

for _, row in eval_df.iterrows():
    q = row['question']
    cat = row['category']
    trace = engine.run_crag_pipeline(q, top_k=3)
    results.append({
        'Question': q[:45] + '...',
        'Category': cat,
        'Initial Verdict': trace['retrieval_grade'],
        'Rewritten?': trace['was_rewritten'],
        'Final Status': trace['status'],
        'Verified Chunks': len(trace['verified_chunks'])
    })

res_df = pd.DataFrame(results)
res_df


## 3. Performance & Hallucination Prevention Analytics


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
res_df['Final Status'].value_counts().plot(kind='bar', color='#1E3A8A', edgecolor='black', ax=ax)
ax.set_title('CRAG Pipeline Outcomes on Benchmark Dataset')
ax.set_xlabel('Outcome')
ax.set_ylabel('Number of Cases')
plt.xticks(rotation=0)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print('Benchmark Summary:')
print(f"- Direct In-Domain Answered: {len(res_df[(res_df['Category'] == 'direct_indomain') & (res_df['Final Status'] == 'SUCCESS')])}")
print(f"- Ambiguous Queries Rewritten & Retrieved: {len(res_df[(res_df['Category'] == 'ambiguous_indomain') & (res_df['Rewritten?'] == True)])}")
print(f"- Out-of-Domain Correctly Refused: {len(res_df[(res_df['Category'] == 'out_of_domain') & (res_df['Final Status'] == 'UNANSWERED')])}")
